# LeetCode #1384: Movie Rating

https://leetcode.com/problems/movie-rating/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(n^2)$ | $O(n)$ |
| **Optimal: Dual Hash Map Aggregation ★** | $O(n)$ | $O(n)$ |

---

## Understanding the Methods

### Brute Force
For each user/movie, scan all ratings to count or sum — nested loops produce $O(n^2)$ comparisons.

### Optimal: Dual Hash Map Aggregation ★
In a single pass over all ratings, maintain (1) a `userRatings` map counting how many movies each user rated, and (2) a `movieAvg` map accumulating the February-2020 rating sums and counts per movie. After the pass, scan each map once to find the top user (alphabetical on tie) and the highest-average movie (alphabetical on tie).

**Why this is better than Brute Force:** Collapses $O(n^2)$ counting into one $O(n)$ aggregation pass plus $O(u + m)$ map scans, where $u$ and $m$ are the number of unique users and movies.

**Constraints:**
* Each rating entry has: `movie_id`, `user_id`, `rating` (1–10), `created_at` (date string `"YYYY-MM-DD"`)
* Break ties by name/title alphabetically (lexicographically smallest)
* At least one user rated a movie; at least one movie was rated in February 2020


## Solutions

### C#

In [ ]:
public class Solution {
    // Rating entry mirrors a database row passed as a struct/tuple
    public record Rating(int MovieId, string MovieTitle, int UserId, string UserName,
                         int Score, string CreatedAt);

    public (string TopUser, string TopMovie) MovieRating(IList<Rating> ratings) {
        var userCount  = new Dictionary<string, int>();
        var febSum     = new Dictionary<string, long>();
        var febCount   = new Dictionary<string, int>();
        var movieTitle = new Dictionary<int, string>();

        foreach (var r in ratings) {
            movieTitle[r.MovieId] = r.MovieTitle;

            // Tally every rating toward that user's total count
            userCount[r.UserName] = userCount.GetValueOrDefault(r.UserName, 0) + 1;

            // Accumulate sum/count for Feb-2020 ratings only
            if (r.CreatedAt.StartsWith("2020-02")) {
                febSum[r.MovieTitle]   = febSum.GetValueOrDefault(r.MovieTitle, 0L) + r.Score;
                febCount[r.MovieTitle] = febCount.GetValueOrDefault(r.MovieTitle, 0) + 1;
            }
        }

        // Top user: most ratings; alphabetically first on tie
        string topUser = userCount
            .OrderByDescending(kv => kv.Value)
            .ThenBy(kv => kv.Key)
            .First().Key;

        // Top movie: highest average rating in Feb 2020; alphabetically first on tie
        string topMovie = febSum.Keys
            .OrderByDescending(m => (double)febSum[m] / febCount[m])
            .ThenBy(m => m)
            .First();

        return (topUser, topMovie);
    }
}

### Python

In [ ]:
class Solution:
    def movieRating(self, ratings: list[dict]) -> tuple[str, str]:
        """
        ratings: list of dicts with keys movie_id, movie_title,
                 user_id, user_name, rating, created_at (YYYY-MM-DD).
        Returns (top_user, top_movie_feb2020).
        """
        from collections import defaultdict

        user_count: dict[str, int] = defaultdict(int)
        feb_sum:    dict[str, int] = defaultdict(int)
        feb_cnt:    dict[str, int] = defaultdict(int)

        for r in ratings:
            # Count every rating toward the user's total
            user_count[r['user_name']] += 1

            # Accumulate Feb-2020 sums for the per-movie average
            if r['created_at'].startswith('2020-02'):
                feb_sum[r['movie_title']] += r['rating']
                feb_cnt[r['movie_title']] += 1

        # Most active user; lexicographically smallest name breaks ties
        top_user = max(user_count, key=lambda u: (user_count[u], [-ord(c) for c in u]))
        # Simpler tie-break: sort by (-count, name) and take first
        top_user = sorted(user_count, key=lambda u: (-user_count[u], u))[0]

        # Highest average rating in Feb 2020; lexicographically smallest title on tie
        top_movie = sorted(
            feb_sum,
            key=lambda m: (-feb_sum[m] / feb_cnt[m], m)
        )[0]

        return top_user, top_movie

### Go

In [ ]:
import "sort"

// Rating represents one row of the movie-ratings table
type Rating struct {
    MovieTitle string
    UserName   string
    Score      int
    CreatedAt  string // "YYYY-MM-DD"
}

func movieRating(ratings []Rating) (string, string) {
    userCount := map[string]int{}
    febSum    := map[string]int{}
    febCnt    := map[string]int{}

    for _, r := range ratings {
        // Every rating increments the user's count
        userCount[r.UserName]++

        // Only February-2020 ratings count toward the movie average
        if len(r.CreatedAt) >= 7 && r.CreatedAt[:7] == "2020-02" {
            febSum[r.MovieTitle] += r.Score
            febCnt[r.MovieTitle]++
        }
    }

    // Find user with most ratings (alphabetically first on tie)
    users := make([]string, 0, len(userCount))
    for u := range userCount {
        users = append(users, u)
    }
    sort.Slice(users, func(i, j int) bool {
        if userCount[users[i]] != userCount[users[j]] {
            return userCount[users[i]] > userCount[users[j]]
        }
        return users[i] < users[j]
    })
    topUser := users[0]

    // Find movie with highest Feb-2020 average (alphabetically first on tie)
    movies := make([]string, 0, len(febSum))
    for m := range febSum {
        movies = append(movies, m)
    }
    sort.Slice(movies, func(i, j int) bool {
        avgI := float64(febSum[movies[i]]) / float64(febCnt[movies[i]])
        avgJ := float64(febSum[movies[j]]) / float64(febCnt[movies[j]])
        if avgI != avgJ {
            return avgI > avgJ
        }
        return movies[i] < movies[j]
    })
    topMovie := movies[0]

    return topUser, topMovie
}

### Rust

In [ ]:
use std::collections::HashMap;

struct Rating {
    movie_title: String,
    user_name:   String,
    score:       i32,
    created_at:  String, // "YYYY-MM-DD"
}

fn movie_rating(ratings: Vec<Rating>) -> (String, String) {
    let mut user_count: HashMap<String, i32> = HashMap::new();
    let mut feb_sum:    HashMap<String, i32> = HashMap::new();
    let mut feb_cnt:    HashMap<String, i32> = HashMap::new();

    for r in &ratings {
        // Tally every rating toward the user's total
        *user_count.entry(r.user_name.clone()).or_insert(0) += 1;

        // Only Feb-2020 ratings contribute to movie averages
        if r.created_at.starts_with("2020-02") {
            *feb_sum.entry(r.movie_title.clone()).or_insert(0) += r.score;
            *feb_cnt.entry(r.movie_title.clone()).or_insert(0) += 1;
        }
    }

    // User with the highest count; lexicographically smallest name breaks ties
    let mut users: Vec<_> = user_count.iter().collect();
    users.sort_by(|a, b| b.1.cmp(a.1).then(a.0.cmp(b.0)));
    let top_user = users[0].0.clone();

    // Movie with the highest Feb-2020 average; alphabetical tiebreak
    let mut movies: Vec<_> = feb_sum.keys().collect();
    movies.sort_by(|a, b| {
        let avg_a = feb_sum[*a] as f64 / feb_cnt[*a] as f64;
        let avg_b = feb_sum[*b] as f64 / feb_cnt[*b] as f64;
        avg_b.partial_cmp(&avg_a).unwrap().then(a.cmp(b))
    });
    let top_movie = movies[0].clone();

    (top_user, top_movie)
}

## Example Scenarios

### 1. Common Case
**Input:** 3 users rated 4 movies; User "Daniel" rated 3, "Monica" rated 2, "Maria" rated 2. "Frozen2" has the highest Feb-2020 average (4.5).
`userCount`: Daniel→3, Monica→2, Maria→2. Top user by count: **Daniel**. Feb averages: Frozen2→4.5, Joker→3.5. Top movie: **Frozen2**.

### 2. Slightly Complex
**Input:** Two users both rated 2 movies: "Alice" and "Bob". Both movies rated in Feb 2020 with identical averages.
Alphabetical tiebreaker: **Alice** wins for top user; the movie coming first alphabetically wins for top movie. Confirms both tiebreakers activate in one query.

### 3. Edge Case: Time Factor
**Input:** $n = 10^6$ ratings, 1000 unique users, 1000 unique movies
Single pass over $10^6$ entries fills both maps. Map scans touch at most $10^3$ users and $10^3$ movies. Total $O(n + u \log u + m \log m) \approx O(n)$ — dominantly linear in the number of ratings.

### 4. Edge Case: Space Factor
**Input:** All $n = 10^6$ ratings are from distinct user IDs, each rating a distinct movie
`userCount` and `febSum`/`febCnt` each hold up to $10^6$ entries — $O(n)$ space in the worst case. The hash maps themselves are the dominant allocation.

### 5. Almost-Impossible but Plausible
**Input:** One user "Zara" rated 5 movies; another user "Aaron" also rated 5. Only "Zara's Movie" was rated in Feb 2020 (single rating of 10).
Top user tiebreak: Aaron < Zara alphabetically → **Aaron**. Top Feb-2020 movie: only one candidate → **Zara's Movie**. Confirms alphabetical tiebreak for users and single-entry Feb-2020 map.
